## OpenAI — RAG investigation

This notebook is to understand this feature to apply this understanding to address an issue in Dimagi Open Chat Studio for the Remote Index feature. https://deepwiki.com/dimagi/open-chat-studio/7-document-collections-and-rag#remoteindexmanager-openai

File search is a tool available in the Responses API. 
It enables models to retrieve information in a knowledge base of previously uploaded files through semantic and keyword search. 
This is a hosted tool managed by OpenAI
When the model decides to use it, it will automatically call the tool, retrieve information from your files, and return an output.
https://developers.openai.com/api/docs/guides/tools-file-search

Adding a file to a vector store kicks off asynchronous ingestion/embedding, so Step 2 polls until the file status is `completed` before Step 3 queries it.

### Step 1 - Upload a file to the File API

In [ ]:
from openai import OpenAI

client = OpenAI()

# Function to upload into OpenAI's file storage from a local file


def create_file(client, local_file_path):
    with open(local_file_path, "rb") as file_content:
        result = client.files.create(file=file_content, purpose="assistants")
    print(result.id)
    return result.id


# use example files in repo
file_id = create_file(client, "../docs/Example KBase doc.pdf")

### Step 2 - Create OpenAI vector store and upload the file to it

In [ ]:
kbase_vector_store = client.vector_stores.create(name="knowledge_base")
print(kbase_vector_store.id)

# create_and_poll blocks until the file finishes ingestion/embedding (status
# reaches "completed" or "failed"), avoiding a race with the file_search query below.
result = client.vector_stores.files.create_and_poll(
    vector_store_id=kbase_vector_store.id, file_id=file_id
)
print(result)

if result.status != "completed":
    raise RuntimeError(f"Vector store file ingestion did not complete: {result.status}")

### Step 3 - include the file_search tool & vector stores in which to search.

In [ ]:
my_vector_store_id = result.vector_store_id

response = client.responses.create(
    model="gpt-4.1-mini",
    input="Concisely summarize the document in 3 bullet points.",
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [my_vector_store_id],
            "max_num_results": 2,
        }
    ],
)
# print(response)
print(response.output_text)

### Test if both vector stores are used in file search

File in Vector Store 1 says “We love dogs” , when i ask the question “what do we love, the answer is: we love dogs”
When i switch the Vector Store (File in VS 2 says: “We love cats”) the answer is “we love cats”.
My expectation was if I add to vector stores, that it says “we love both”. It does, when i put both files in one Vector store, but not when i have 2 vector stores

In [ ]:
# Upload one file per vector store
dogs_file_id = create_file(client, "../docs/File-dogs.txt")
cats_file_id = create_file(client, "../docs/File-cats.txt")

# Create two separate vector stores, one file each
vector_store_dogs = client.vector_stores.create(name="dogs_store")
vector_store_cats = client.vector_stores.create(name="cats_store")
print(vector_store_dogs.id, vector_store_cats.id)

result_dogs = client.vector_stores.files.create_and_poll(
    vector_store_id=vector_store_dogs.id, file_id=dogs_file_id
)
result_cats = client.vector_stores.files.create_and_poll(
    vector_store_id=vector_store_cats.id, file_id=cats_file_id
)

for result in (result_dogs, result_cats):
    if result.status != "completed":
        raise RuntimeError(
            f"Vector store file ingestion did not complete: {result.status}"
        )

# Query file_search with both vector stores listed
response = client.responses.create(
    model="gpt-4.1-mini",
    input="What do we love?",
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store_dogs.id, vector_store_cats.id],
            "max_num_results": 2,
        }
    ],
)
print(response.output_text)

### Test - can we create and search 3 vector stores for OpenAI
Use the 3 text files for similar code to above

In [ ]:
import openai

# Upload one file per vector store
dogs_file_id = create_file(client, "../docs/File-dogs.txt")
cats_file_id = create_file(client, "../docs/File-cats.txt")
pigs_file_id = create_file(client, "../docs/File-pigs.txt")

# Create three separate vector stores, one file each
vector_store_dogs = client.vector_stores.create(name="dogs_store")
vector_store_cats = client.vector_stores.create(name="cats_store")
vector_store_pigs = client.vector_stores.create(name="pigs_store")
print(vector_store_dogs.id, vector_store_cats.id, vector_store_pigs.id)

result_dogs = client.vector_stores.files.create_and_poll(
    vector_store_id=vector_store_dogs.id, file_id=dogs_file_id
)
result_cats = client.vector_stores.files.create_and_poll(
    vector_store_id=vector_store_cats.id, file_id=cats_file_id
)
result_pigs = client.vector_stores.files.create_and_poll(
    vector_store_id=vector_store_pigs.id, file_id=pigs_file_id
)

for result in (result_dogs, result_cats, result_pigs):
    if result.status != "completed":
        raise RuntimeError(
            f"Vector store file ingestion did not complete: {result.status}"
        )

# Query file_search with all three vector stores listed.
# Expectation: the API rejects more than 2 vector_store_ids for file_search, so
# catch the exception rather than letting it bubble up.
try:
    response = client.responses.create(
        model="gpt-4.1-mini",
        input="What do we love?",
        tools=[
            {
                "type": "file_search",
                "vector_store_ids": [
                    vector_store_dogs.id,
                    vector_store_cats.id,
                    vector_store_pigs.id,
                ],
                "max_num_results": 2,
            }
        ],
    )
    print(response.output_text)
except openai.OpenAIError as e:
    print(f"Request failed as expected: {e}")